In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from IPython.display import display
import copy

from encoder import MoiraiEncoder
from heads import *
from models import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [64, 32, 16, 8]

# DataLoaders : Garde un Batch Size modéré (16) pour le Full FT
BATCH_SIZE = 256 
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

PATCHES_COUNTS = {64: 2, 32: 3, 16: 4, 8: 6}

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class HeterogeneousMultiScaleWrapper(nn.Module):
    """
    Wrapper Multi-Échelles qui utilise des encodeurs PRÉ-ENTRAÎNÉS et potentiellement HÉTÉROGÈNES
    (ex: Full FT pour 64, DoRA pour 32, etc.).
    """
    def __init__(self, best_encoders_dict, head_class, head_kwargs, num_vars=6, remove_last_patch=False):
        super().__init__()
        self.patch_sizes = [64, 32, 16, 8] 
        self.num_vars = num_vars
        self.remove_last_patch = remove_last_patch
        
        # On utilise ModuleDict avec des clés en string ("64", "32"...) pour stocker les encodeurs
        self.encoders = nn.ModuleDict({
            str(p): best_encoders_dict[p] for p in self.patch_sizes
        })
        
        # Initialisation de la tête multi-échelles (Flatten, Sequential ou Parallel)
        self.head = head_class(**head_kwargs)

    def forward(self, target, obs, pad):
        features_list = []
        for p in self.patch_sizes:
            # On utilise l'encodeur spécialiste du patch
            Z = self.encoders[str(p)](target, obs, pad)
            
            if self.remove_last_patch:
                B, S, F = Z.shape
                P = S // self.num_vars
                Z_reshaped = Z.view(B, self.num_vars, P, F)
                Z_no_mask = Z_reshaped[:, :, :-1, :]
                Z = Z_no_mask.reshape(B, -1, F)
                
            features_list.append(Z)
        
        concat_features = torch.cat(features_list, dim=1)
        return self.head(concat_features)

In [ ]:
best_encoders_dict = {}
summary_results = []

ranks_to_test = [1, 4,  64 , 1024]

encoders = []
for p in PATCH_SIZES:
    print(f"\n{'='*60}\n🏆 SÉLECTION DU CHAMPION POUR LE PATCH {p} 🏆\n{'='*60}")
    
    best_acc_for_patch = 0.0
    best_model_for_patch = None
    best_method_name = ""
    
    # ----------------------------------------------------
    # 1. Candidat : Full Fine-Tuning
    # ----------------------------------------------------
    print(f"🔄 Test: Full Fine-Tuning")
    model_full, acc_full = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": p, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=[0.01,0.05], lr_grid=[1e-4]
    )
    summary_results.append({"Patch": p, "Méthode": "Full FT", "Rank": "-", "Test Acc": acc_full})
    
    if acc_full > best_acc_for_patch:
        best_acc_for_patch = acc_full
        best_model_for_patch = copy.deepcopy(model_full)
        best_method_name = "Full FT"
    model = copy.deepcopy(model_full)
    encoders.append(model.model.encoder)

    # ----------------------------------------------------
    # 2. Candidats : PEFT (LoRA, DoRA, AdaLoRA)
    # ----------------------------------------------------
    for method in ['AdaLora']:#["LoRA", "DoRA", "AdaLoRA"]:
        for r in ranks_to_test:
            print(f"🔄 Test: {method} (Rank {r})")
            
            use_dora = (method == "DoRA")
            use_adalora = (method == "AdaLoRA")
            
            model_peft, acc_peft = universal_grid_search(
                model_class=LoraHeadWrapper,
                model_kwargs={
                    "head_class": MeanPoolingClassifier,
                    "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                    "patch_size": p, "num_vars": NUM_VARS, "size": SIZE, 
                    "remove_last_patch": False, "lora_r": r,
                    "use_dora": use_dora, "use_adalora": use_adalora
                },
                train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
                wd_grid=[0.01,0.05], lr_grid=[1e-4]
            )
            summary_results.append({"Patch": p, "Méthode": method, "Rank": r, "Test Acc": acc_peft})
            
            if acc_peft > best_acc_for_patch:
                best_acc_for_patch = acc_peft
                best_model_for_patch = copy.deepcopy(model_peft)
                best_method_name = f"{method} (r={r})"
            encoders.append(copy.deepcopy(model_peft).encoder)

    # ----------------------------------------------------
    # Sauvegarde de l'encodeur gagnant pour ce patch
    # ----------------------------------------------------
    print(f"⭐ GAGNANT DU PATCH {p} : {best_method_name} avec Acc={best_acc_for_patch:.4f}")
    # On détache l'encodeur de la tête MeanPooling et on le stocke
    best_encoders_dict[p] = best_model_for_patch.encoder


🏆 SÉLECTION DU CHAMPION POUR LE PATCH 64 🏆
🔄 Test: Full Fine-Tuning
LR=0.0001 | WD=0.01
 Early stopping : 10
Val Loss: 1.1642
LR=0.0001 | WD=0.05
 Early stopping : 11
Val Loss: 1.1501
Acc on Test Set : 0.6221

🔄 Test: AdaLora (Rank 1)
LR=0.0001 | WD=0.01
 Early stopping : 58
Val Loss: 1.1225
LR=0.0001 | WD=0.05
 Early stopping : 60
Val Loss: 1.1114
Acc on Test Set : 0.6314

🔄 Test: AdaLora (Rank 4)
LR=0.0001 | WD=0.01
 Early stopping : 35
Val Loss: 1.1445
LR=0.0001 | WD=0.05
 Early stopping : 37
Val Loss: 1.1280
Acc on Test Set : 0.6290

🔄 Test: AdaLora (Rank 64)
LR=0.0001 | WD=0.01
 Early stopping : 18
Val Loss: 1.1412
LR=0.0001 | WD=0.05
 Early stopping : 15
Acc on Test Set : 0.6306

🔄 Test: AdaLora (Rank 1024)
LR=0.0001 | WD=0.01


In [ ]:
df_phase1 = pd.DataFrame(summary_results)
print("RÉCAPITULATIF DE TOUTES LES EXPÉRIENCES (PHASE 1):")
display(df_phase1.pivot(index=["Patch", "Méthode"], columns="Rank", values="Test Acc").round(4))

NameError: name 'pd' is not defined

In [ ]:
print(f"\n{'='*60}\n🚀 PHASE 2 : ENTRAÎNEMENT DES TÊTES MULTI-ÉCHELLES\n{'='*60}")

# Les différentes approches d'attention pour l'agrégation
experiences_multi = [
    ("Baseline: Flatten Multi-Scale", MultiScaleMeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS}),
    ("Sequential (Cascade) | Indépendant", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 64, "shared_layer": True}),
    ("Parallèle (Concat) | Indépendant", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 64, "shared_layer": True})
]

df_final = pd.DataFrame(index=[exp[0] for exp in experiences_multi], columns=["Test Accuracy"])
df_final.index.name = "Agrégation Multi-Scale (Avec Meilleurs Encodeurs)"

for arch_name, head_class, head_kwargs in experiences_multi:
    print(f"🔄 Test en cours : {arch_name}")
    
    _, test_acc = universal_grid_search(
        model_class=HeterogeneousMultiScaleWrapper, # <--- On utilise le NOUVEAU Wrapper
        model_kwargs={
            "best_encoders_dict": best_encoders_dict, # On injecte notre dictionnaire de champions
            "head_class": head_class,
            "head_kwargs": head_kwargs,
            "num_vars": NUM_VARS,
            "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=[0.05], 
        lr_grid=[1e-4],
        verbose = True        # Taux d'apprentissage rapide pour la NOUVELLE tête d'attention
    )
    
    df_final.loc[arch_name, "Test Accuracy"] = test_acc

print("\n" + "="*50)
print("🏆 RÉSULTATS GLOBAUX MULTI-SCALE 🏆")
print("="*50)
display(df_final.astype(float).round(4))


🚀 PHASE 2 : ENTRAÎNEMENT DES TÊTES MULTI-ÉCHELLES
🔄 Test en cours : Baseline: Flatten Multi-Scale
LR=0.0001 | WD=0.05
1.5114090588034652
1.261203632122133
1.152794055822419
1.1486115523470126
1.1160855991084402
1.1235671489219354
1.109649105769832
1.0719831387201946
1.1203987191363078
1.1901652551278836
1.2682480598852885
1.2929016826598625
1.3483100664324876
 Early stopping : 12
Val Loss: 1.0720
Acc on Test Set : 0.6415

🔄 Test en cours : Sequential (Cascade) | Indépendant
LR=0.0001 | WD=0.05
1.4379144829463184
1.2922234399531916
1.3224616574078072
1.3931989543806247
1.3372609188886193
1.4739677712200134
1.487326858489494
 Early stopping : 6
Val Loss: 1.2922
Acc on Test Set : 0.6099

🔄 Test en cours : Parallèle (Concat) | Indépendant
LR=0.0001 | WD=0.05
1.6701180479390834
1.4632718824758761
1.4346529284143836
